In [2]:
import cv2
import mediapipe as mp
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional,LSTM, Dense,Input,Flatten, GRU

from glob import glob
import os
from pathlib import Path

from tqdm.notebook import tqdm

from numpy import  argmax, array, expand_dims 
import numpy as np

import utils


In [3]:
# Initialize Mediapipe Pose model
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils  
pose = mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7)


mp_utils = utils.MediapipeUtils(mp_pose, mp_drawing)

In [4]:
# Specify the video file(s)
DATA_DIR = glob("Dataset/*/*/*.mp4")
DATA_PATH = "MP_Data"

# Landmarks to extract (based on Mediapipe Pose Landmarks)
# landmark_indices = [11, 12, 13, 14, 15, 16]

angle_landmarks = ((12,14),
                   (14,16),
                   (11,13),
                   (13,15))

labels = [i.split("\\")[-1] for i in glob("Dataset\*\*")]

labels

['Blind',
 'Deaf',
 'Flat',
 'Happy',
 'Poor',
 'Quiet',
 'Rich',
 'sad',
 'Slow',
 'Thick']

In [5]:
try:
    # os.mkdir('MP_data')
    for i in range(len(labels)):
        os.mkdir(f'MP_data/{labels[i]}')
except:
    print("Directory already exists")

Directory already exists


In [6]:
# Process each video
for video_path in tqdm(DATA_DIR):
    data = []
    cap = cv2.VideoCapture(video_path)
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the frame to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Process the image and detect pose landmarks
        results = pose.process(image)

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Extract X, Y, Z coordinates for specified landmarks
            # keypoints = []
            angles_keypoint = []
            # for idx in landmark_indices:
            #     lm = landmarks[idx]
            #     keypoints.extend([lm.x, lm.y, lm.z])

            for idx in angle_landmarks:
                p1 = landmarks[idx[0]]                
                p2 = landmarks[idx[1]]    
                
                vector = np.array([p2.x - p1.x, p2.y - p1.y, p2.z - p1.z])
                            
                ang = mp_utils.calculate_angles(vector)
                angles_keypoint.extend(ang)
            
            # Calculate angles between vectors
            # Left arm: vectors 15-13 and 13-11
            left_wrist = np.array([landmarks[15].x, landmarks[15].y, landmarks[15].z])
            left_elbow = np.array([landmarks[13].x, landmarks[13].y, landmarks[13].z])
            left_shoulder = np.array([landmarks[11].x, landmarks[11].y, landmarks[11].z])

            vector_left_wrist_elbow = left_wrist - left_elbow
            vector_left_elbow_shoulder = left_elbow - left_shoulder

            angle_left_arm = mp_utils.calculate_angles_between(vector_left_wrist_elbow, vector_left_elbow_shoulder)

            # Right arm: vectors 16-14 and 14-12
            right_wrist = np.array([landmarks[16].x, landmarks[16].y, landmarks[16].z])
            right_elbow = np.array([landmarks[14].x, landmarks[14].y, landmarks[14].z])
            right_shoulder = np.array([landmarks[12].x, landmarks[12].y, landmarks[12].z])

            vector_right_wrist_elbow = right_wrist - right_elbow
            vector_right_elbow_shoulder = right_elbow - right_shoulder

            angle_right_arm = mp_utils.calculate_angles_between(vector_right_wrist_elbow, vector_right_elbow_shoulder)
            
            # Distance between points 15 and 16 (hands)
            left_hand = landmarks[15]
            right_hand = landmarks[16]
            centre = landmarks[0]
            
            distance_hands = mp_utils.calculate_distance(left_hand, right_hand)
            distance_left_hand_centre = mp_utils.calculate_distance(left_hand, centre)
            distance_right_hand_centre = mp_utils.calculate_distance(right_hand, centre)
                        
            
            # Combine all features
            features = angles_keypoint + [angle_left_arm, angle_right_arm, distance_hands, distance_left_hand_centre, distance_right_hand_centre]
            data.append(features)

    data = np.array(data)
    path = video_path.split("/")
    
    # Getting Current action of the video
    action = video_path.split("\\")[-2]
    
    #Get the sequence number
    sequence = video_path.split("\\")[-1][:-4]
    
    npy_path = os.path.join(DATA_PATH, action, sequence)
    print(npy_path)
    
    np.save(npy_path, data)

    cap.release()

pose.close()


  0%|          | 0/110 [00:00<?, ?it/s]

MP_Data\Blind\1-1
MP_Data\Blind\1-2
MP_Data\Blind\1-3
MP_Data\Blind\1-4
MP_Data\Blind\1-5
MP_Data\Blind\1-6
MP_Data\Blind\1-7
MP_Data\Blind\1-8
MP_Data\Deaf\Deaf-1
MP_Data\Deaf\Deaf-2
MP_Data\Deaf\Deaf-3
MP_Data\Deaf\Deaf-4
MP_Data\Deaf\Deaf-5
MP_Data\Deaf\Deaf-6
MP_Data\Deaf\Deaf-7
MP_Data\Deaf\Deaf-8
MP_Data\Flat\Flat-1
MP_Data\Flat\Flat-2
MP_Data\Flat\Flat-3
MP_Data\Flat\Flat-4
MP_Data\Flat\Flat-5
MP_Data\Flat\Flat-6
MP_Data\Flat\Flat-7
MP_Data\Flat\Flat-8
MP_Data\Happy\Happy-1
MP_Data\Happy\Happy-10
MP_Data\Happy\Happy-11
MP_Data\Happy\Happy-12
MP_Data\Happy\Happy-13
MP_Data\Happy\Happy-14
MP_Data\Happy\Happy-15
MP_Data\Happy\Happy-2
MP_Data\Happy\Happy-3
MP_Data\Happy\Happy-4
MP_Data\Happy\Happy-5
MP_Data\Happy\Happy-6
MP_Data\Happy\Happy-7
MP_Data\Happy\Happy-8
MP_Data\Happy\Happy-9
MP_Data\Poor\Poor-1
MP_Data\Poor\Poor-2
MP_Data\Poor\Poor-3
MP_Data\Poor\Poor-4
MP_Data\Poor\Poor-5
MP_Data\Poor\Poor-6
MP_Data\Poor\Poor-7
MP_Data\Poor\Poor-8
MP_Data\Quiet\quiet-1
MP_Data\Quiet\quie